# Анализ эксперимента работы модели по подстройке коэффициентов Kp и Ki обученой 30 октября 2025

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from nn_laser_stabilizer.utils.paths import get_experiment_dir

EXPERIMENT_NAME = "td3_inference_async"
EXPERIMENT_DATE = "2025-11-11"
EXPERIMENT_TIME = "15-03-10"

EXPERIMENT_DIR = get_experiment_dir(
    experiment_name=EXPERIMENT_NAME,
    experiment_date=EXPERIMENT_DATE,
    experiment_time=EXPERIMENT_TIME,
)
ENV_LOG_PATH = EXPERIMENT_DIR / "env_logs" / "env.log"
TRAIN_LOG_PATH = EXPERIMENT_DIR / "train_async.log"
TRAIN_LOGS_PATH = EXPERIMENT_DIR / "train_logs" / "train.log"
CONNECTION_LOG_PATH = EXPERIMENT_DIR / "connection_logs" / "connection.log"

## Анализ логов окружения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_env_log

# Сопоставление сырых полей лога с историческими именами колонок блокнота.
ENV_COLUMNS = {
    "step": "Step",
    "kp": "Kp",
    "ki": "Ki",
    "kd": "Kd",
    "delta_kp_norm": "Delta Kp",
    "delta_ki_norm": "Delta Ki",
    "error_mean": "Error mean",
    "error_std": "Error std",
    "error_mean_norm": "Error mean norm",
    "error_std_norm": "Error std norm",
    "reward": "Reward",
}

env_df = (
    parse_env_log(ENV_LOG_PATH)
    .rename(columns=ENV_COLUMNS)[list(ENV_COLUMNS.values())]
    .dropna()  # как старый парсер: только записи со всеми полями
    .reset_index(drop=True)
)
print(f"Загружено {len(env_df)} записей из логов окружения")

In [ ]:
print("=== Статистика по данным окружения ===")
print(env_df.describe())

In [ ]:
columns_to_plot = [col for col in env_df.columns if col != 'Step']

for col in columns_to_plot:
    plt.figure(figsize=(10, 4))
    plt.plot(env_df['Step'], env_df[col], alpha=0.8, linewidth=0.8, label=col)
    
    plt.title(f'{col} over Steps')
    plt.xlabel('Step')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=env_df['Kp'], y=env_df['Error std'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error Std')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=env_df['Kp'], y=env_df['Error mean'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error mean')
cur_ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=env_df['Ki'], y=env_df['Error std'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error Std')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=env_df['Ki'], y=env_df['Error mean'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error mean')
cur_ax.grid(True, alpha=0.3)

In [ ]:
corr_matrix = env_df[['Kp', 'Error mean', 'Error std', 'Reward']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f");

In [ ]:
corr_matrix = env_df[['Ki', 'Error mean', 'Error std', 'Reward']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f");

## Анализ состояния установки


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_connection_log

# parse_connection_log отдаёт сырые записи SEND/READ (тип — в колонке 'event').
# Историческое поведение: i-я посылка (SEND) спаривается с i-м чтением (READ).
conn_raw = parse_connection_log(CONNECTION_LOG_PATH)

send_df = conn_raw[conn_raw["event"] == "SEND"][
    ["kp", "ki", "kd", "control_min", "control_max"]
].reset_index(drop=True)
send_df["step"] = range(len(send_df))

read_df = conn_raw[conn_raw["event"] == "READ"][
    ["process_variable", "control_output"]
].reset_index(drop=True)
read_df["step"] = range(len(read_df))

connection_df = send_df.merge(read_df, on="step", how="left")
print(f"Загружено {len(connection_df)} записей из логов соединения")
print(f"Диапазон шагов: {connection_df['step'].min()} - {connection_df['step'].max()}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=0.8)
plt.axhline(y=1200, color='r', linestyle='--', label='Setpoint')

plt.title('Process Variable')
plt.xlabel('Step')
plt.ylabel('Process Variable')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=0.8)

plt.title('Control Output')
plt.xlabel('Step')
plt.ylabel('Control Output')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(connection_df['step'], connection_df['kp'], 'r-', alpha=0.7, linewidth=0.8, label='Kp')
plt.plot(connection_df['step'], connection_df['ki'], 'g-', alpha=0.7, linewidth=0.8, label='Ki')
plt.plot(connection_df['step'], connection_df['kd'], 'b-', alpha=0.7, linewidth=0.8, label='Kd')

plt.title('PID Coefficients')
plt.xlabel('Step')
plt.ylabel('Coefficient Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
corr_matrix = connection_df[['kp', 'ki', 'control_output', 'process_variable']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f");